# 04 · Reciclo de licor madre, evaporación al vacío y purga (estacionario)

Resuelve el cristalizador A3110 con reciclo usando **las mismas ecuaciones del modelo dinámico** (`msmpr_core.plant`): el estacionario es `rhs = 0`.

**Advertencia.** Con los parámetros actuales (`C*_ref`, alimentación de KNO3, cinética) los números son *placeholders*: sirven para ver **tendencias y la estructura del balance**, no para citar cifras de planta. Se listan abajo.

**Resultado estructural importante.** Con volumen fijo, la purga la fija el balance de agua (alimentación − evaporación). La fracción `alpha` no cambia cuánto se pierde por purga: cambia cuánto licor circula (tiempo de residencia efectivo).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))           # raíz del repo (donde está msmpr_core)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from msmpr_core.params import make_params, placeholders, PROVENANCE
from msmpr_core.plant import Plant, simulate
from msmpr_core import control as C
os.makedirs("../docs/fig", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

In [ ]:
p = make_params()
print("Parámetros que siguen siendo supuestos:", ", ".join(placeholders()))
pl = Plant(p)

## 0. Alineación con la base del notebook de calibración
Los `kG`, `kB`, `Kd` de `calibracion_do_crystal.ipynb` solo valen sobre su base. Esta tabla muestra qué difiere de la base con fuentes (`literatura`) y cuáles diferencias condicionan esos parámetros. Para usar valores calibrados: `make_params("calibracion")` + `apply_calibration`.

In [ ]:
from msmpr_core.params import diff_bases
diff_bases()

## 1. Barrido de la fracción de purga α (T = 40 °C, V fijo)

In [ ]:
rows = []
for a in [1.0, 0.75, 0.5, 0.35, 0.25, 0.15, 0.10, 0.05]:
    x, u, d = pl.steady_state(alpha=a)
    m = pl.metrics(x, u, d)
    rows.append({"alpha": a, "reciclo_m3_h": m["recycle_m3_h"], "purga_m3_h": m["purge_m3_h"], "tau_h": m["tau_h"],
                 "evap_t_h": m["evap_t_h"], "sigma": m["sigma"], "Y_%": m["recovery_pct"], "Y_cierre": m["Y_cierre"],
                 "d50_um": m["d50_um"], "ClO4_licor_g_L": m["ClO4_liquor_g_L"], "ppm_ClO4": m["ppm_ClO4"]})
df_alpha = pd.DataFrame(rows).round(4)
df_alpha

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].plot(df_alpha.alpha, df_alpha.purga_m3_h, "o-"); ax[0].set(xlabel="α (fracción purgada)", ylabel="purga [m³/h]", title="La purga no depende de α", ylim=(0, 1.2*df_alpha.purga_m3_h.max()))
ax[1].plot(df_alpha.alpha, df_alpha.d50_um, "o-", color="tab:green"); ax[1].set(xlabel="α", ylabel="d50 [µm]", title="α mueve el tiempo de residencia efectivo")
ax[2].plot(df_alpha.alpha, df_alpha["Y_%"], "o-", color="tab:red"); ax[2].set(xlabel="α", ylabel="recuperación Y [%]", title="Y (placeholders)")
fig.tight_layout(); fig.savefig("../docs/fig/04_alpha.png"); plt.show()

## 2. Palancas reales del ClO₄⁻: dilución y temperatura

In [ ]:
rows = []
for d_ in [0.0, 0.05, 0.10, 0.15, 0.20]:
    for T in [35.0, 40.0, 45.0]:
        q = make_params(feed_dilution_fraction=d_, crystallizer_temperature_C=T)
        pl_ = Plant(q)
        try:
            x, u, dd = pl_.steady_state()
            m = pl_.metrics(x, u, dd)
            rows.append({"dilución": d_, "T_C": T, "Y_%": m["recovery_pct"], "ppm_ClO4": m["ppm_ClO4"], "d50_um": m["d50_um"], "evap_t_h": m["evap_t_h"]})
        except RuntimeError:
            rows.append({"dilución": d_, "T_C": T, "Y_%": np.nan, "ppm_ClO4": np.nan, "d50_um": np.nan, "evap_t_h": np.nan})
df_ops = pd.DataFrame(rows).round(3)
fig, ax = plt.subplots(figsize=(6.2, 4.2))
for T, g in df_ops.groupby("T_C"):
    ax.plot(g["Y_%"], g.ppm_ClO4, "o-", label=f"T = {T:.0f} °C")
ax.set(xlabel="recuperación Y [%]  (maximizar)", ylabel="ClO₄⁻ en producto [ppm]  (minimizar)", title="Conflicto producción–pureza (barrido de dilución)")
ax.legend(); fig.tight_layout(); fig.savefig("../docs/fig/04_conflicto.png"); plt.show()
df_ops

## Lectura
* La dilución baja el ppm de ClO₄ y también la recuperación: es el conflicto del frente de Pareto de la guía (§7).
* Con los placeholders actuales, a dilución alta o T alta casi no cristaliza (Y→0): la alimentación queda bajo la solubilidad supuesta. Falta medir la solubilidad del licor real.
* El ppm usa un `Kd` supuesto y el ClO₄ de alimentación del informe de parámetros (5.56 g/L, base iónica). Hay que validar ambos antes de leer ppm absolutos.